In [1]:
import os
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve
)

# ─────────────────────────────────────────────
# 0. SEED & DEVICE
# ─────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ─────────────────────────────────────────────
# 1. PATHS
# ─────────────────────────────────────────────
BASE_DIR = r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\ranked rf"
RESULT_DIR = os.path.join(os.path.dirname(BASE_DIR), "results_mlp_sfs")
os.makedirs(RESULT_DIR, exist_ok=True)

X_TRAIN_PATH = os.path.join(BASE_DIR, "X_train_rF_DEG_train_lung.csv")
X_TEST_PATH  = os.path.join(BASE_DIR, "X_test_rF_DEG_train_lung.csv")
Y_TRAIN_PATH = os.path.join(BASE_DIR, "y_train_rF_DEG_train.csv")
Y_TEST_PATH  = os.path.join(BASE_DIR, "y_test_rF_DEG_train.csv")


Using device: cpu


In [2]:

# ─────────────────────────────────────────────
# 2. LOAD & CLEAN
# ─────────────────────────────────────────────
def clean_feature_df(df):
    df = df.copy()
    return df.loc[:, ~df.columns.str.contains("^Unnamed")]

def clean_label_df(df):
    df = df.copy()
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    return df.iloc[:, -1].astype(int)

X_train_full = clean_feature_df(pd.read_csv(X_TRAIN_PATH, low_memory=False))
X_test       = clean_feature_df(pd.read_csv(X_TEST_PATH,  low_memory=False))
y_train_full = clean_label_df(pd.read_csv(Y_TRAIN_PATH,  low_memory=False))
y_test       = clean_label_df(pd.read_csv(Y_TEST_PATH,   low_memory=False))


In [3]:

# Column order = RF rank order (rank 0 = best)
gene_names = X_train_full.columns.tolist()
print(f"Total ranked genes: {len(gene_names)}")

# ─────────────────────────────────────────────
# 3. SCALE  (fit on full train, transform test)
# ─────────────────────────────────────────────
scaler = StandardScaler()
X_train_full = pd.DataFrame(
    scaler.fit_transform(X_train_full), columns=gene_names
)
X_test = pd.DataFrame(
    scaler.transform(X_test), columns=gene_names
)

# ─────────────────────────────────────────────
# 4. TRAIN / VAL SPLIT
# ─────────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2, stratify=y_train_full,
    random_state=RANDOM_STATE
)


Total ranked genes: 211


In [4]:

# ─────────────────────────────────────────────
# 5. DATASET & HELPERS
# ─────────────────────────────────────────────
class GeneExpressionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


def make_loaders(X_tr, X_vl, X_te, y_tr, y_vl, y_te, batch_size=32):
    return (
        DataLoader(GeneExpressionDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True),
        DataLoader(GeneExpressionDataset(X_vl, y_vl), batch_size=batch_size, shuffle=False),
        DataLoader(GeneExpressionDataset(X_te, y_te), batch_size=batch_size, shuffle=False),
    )


def run_epoch(model, loader, criterion, optimizer=None):
    model.train() if optimizer else model.eval()
    total_loss, all_probs, all_labels = 0, [], []

    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        with torch.set_grad_enabled(optimizer is not None):
            logits = model(X_b)
            loss   = criterion(logits, y_b)
            if optimizer:
                optimizer.zero_grad(); loss.backward(); optimizer.step()

        all_probs.extend(torch.sigmoid(logits).detach().cpu().numpy())
        all_labels.extend(y_b.detach().cpu().numpy())
        total_loss += loss.item() * len(X_b)

    return total_loss / len(loader.dataset), np.array(all_labels), np.array(all_probs)


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    sn = recall_score(y_true, y_pred, zero_division=0)
    sp = tn / (tn + fp) if (tn + fp) > 0 else 0
    return {
        "ACC":       accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Sn":        sn,
        "Sp":        sp,
        "F1":        f1_score(y_true, y_pred, zero_division=0),
        "AUC":       roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.0,
        "BA":        (sn + sp) / 2,
    }


def fmt(v): return f"{v * 100:.2f}%"

# ─────────────────────────────────────────────
# 6. MODEL
# ─────────────────────────────────────────────
class MLPDropoutBN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(512, 256),       nn.BatchNorm1d(256), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(True), nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
    def forward(self, x): return self.net(x).squeeze(-1)


In [5]:

# ─────────────────────────────────────────────
# 7. HYPERPARAMS
# ─────────────────────────────────────────────
BATCH_SIZE  = 32
EPOCHS      = 100
PATIENCE    = 15
START_GENE  = 10   # mirror Document 2: ranks[10:]
COLS = ['Gene', 'N_Genes', 'ACC', 'Sn', 'Precision', 'Sp', 'F1', 'AUC', 'BA', 'Threshold']

result_val  = []
result_test = []

pos_count  = y_train.sum()
neg_count  = len(y_train) - pos_count
pos_weight = torch.tensor([neg_count / pos_count], dtype=torch.float32).to(device)


In [7]:

# ─────────────────────────────────────────────
# 8. SFS LOOP
# ─────────────────────────────────────────────
for i in range(START_GENE, len(gene_names)):

    selected = gene_names[:i + 1]          # top-(i+1) RF-ranked genes
    current_gene = gene_names[i]           # the gene added this round

    train_loader, val_loader, test_loader = make_loaders(
        X_train[selected], X_val[selected], X_test[selected],
        y_train, y_val, y_test,
        batch_size=BATCH_SIZE
    )

    # ── Build fresh model ──────────────────────
    model     = MLPDropoutBN(input_dim=i + 1).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # ── Train with early stopping ──────────────
    best_val_auc    = -1
    wait            = 0
    best_weights    = None          # store in RAM, not disk

    for epoch in range(1, EPOCHS + 1):
        run_epoch(model, train_loader, criterion, optimizer)

        _, y_v, p_v = run_epoch(model, val_loader, criterion)
        val_auc = compute_metrics(y_v, p_v)["AUC"]

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            wait         = 0
            best_weights = copy.deepcopy(model.state_dict())   # RAM save
        else:
            wait += 1

        if wait >= PATIENCE:
            break

    # ── Load best weights ──────────────────────
    model.load_state_dict(best_weights)
    model.eval()

    # ── Threshold selection on validation ─────
    _, y_val_true, y_val_prob = run_epoch(model, val_loader, criterion)
    fpr, tpr, thresholds = roc_curve(y_val_true, y_val_prob)
    best_threshold = thresholds[np.argmax(tpr - fpr)]

    val_metrics = compute_metrics(y_val_true, y_val_prob, threshold=best_threshold)

    # ── Test evaluation ────────────────────────
    _, y_test_true, y_test_prob = run_epoch(model, test_loader, criterion)
    test_metrics = compute_metrics(y_test_true, y_test_prob, threshold=best_threshold)

    # ── Log ───────────────────────────────────
    print(
        f"Genes={i+1:3d} | added={current_gene:<20s} | "
        f"val_AUC={val_metrics['AUC']:.4f} | "
        f"test_AUC={test_metrics['AUC']:.4f} | "
        f"thr={best_threshold:.4f}"
    )

    def row(m): return [
        current_gene, i + 1,
        fmt(m["ACC"]), fmt(m["Sn"]), fmt(m["Precision"]),
        fmt(m["Sp"]),  fmt(m["F1"]), fmt(m["AUC"]), fmt(m["BA"]),
        f"{best_threshold:.4f}"
    ]

    result_val.append(row(val_metrics))
    result_test.append(row(test_metrics))

    # ── Save incrementally (safe if interrupted) ──
    pd.DataFrame(result_val,  columns=COLS).to_csv(
        os.path.join(RESULT_DIR, "mlp_bn_sfs_val.csv"),  index=False)
    pd.DataFrame(result_test, columns=COLS).to_csv(
        os.path.join(RESULT_DIR, "mlp_bn_sfs_test.csv"), index=False)

print("\nDone. Results saved to:", RESULT_DIR)

Genes= 11 | added=ENSG00000137502      | val_AUC=0.9193 | test_AUC=0.8895 | thr=0.3639
Genes= 12 | added=ENSG00000177556      | val_AUC=0.9222 | test_AUC=0.8745 | thr=0.6550
Genes= 13 | added=ENSG00000173083      | val_AUC=0.9279 | test_AUC=0.9078 | thr=0.3432
Genes= 14 | added=ENSG00000197459      | val_AUC=0.9248 | test_AUC=0.9167 | thr=0.3634
Genes= 15 | added=ENSG00000103316      | val_AUC=0.9334 | test_AUC=0.9266 | thr=0.2922
Genes= 16 | added=ENSG00000223609      | val_AUC=0.9422 | test_AUC=0.9203 | thr=0.4243
Genes= 17 | added=ENSG00000229124      | val_AUC=0.9334 | test_AUC=0.9187 | thr=0.4891
Genes= 18 | added=ENSG00000213465      | val_AUC=0.9325 | test_AUC=0.9181 | thr=0.5519
Genes= 19 | added=ENSG00000187554      | val_AUC=0.9396 | test_AUC=0.9166 | thr=0.5986
Genes= 20 | added=ENSG00000066185      | val_AUC=0.9294 | test_AUC=0.9241 | thr=0.3890
Genes= 21 | added=ENSG00000230773      | val_AUC=0.9361 | test_AUC=0.9198 | thr=0.3395
Genes= 22 | added=ENSG00000267243      | va

In [9]:
result_val

[['ENSG00000137502',
  11,
  '87.67%',
  '86.75%',
  '91.14%',
  '88.89%',
  '88.89%',
  '91.78%',
  '87.82%',
  '0.4361'],
 ['ENSG00000177556',
  12,
  '86.30%',
  '91.57%',
  '85.39%',
  '79.37%',
  '88.37%',
  '93.04%',
  '85.47%',
  '0.2867'],
 ['ENSG00000173083',
  13,
  '84.25%',
  '77.11%',
  '94.12%',
  '93.65%',
  '84.77%',
  '92.85%',
  '85.38%',
  '0.5308'],
 ['ENSG00000197459',
  14,
  '84.93%',
  '80.72%',
  '91.78%',
  '90.48%',
  '85.90%',
  '92.96%',
  '85.60%',
  '0.5252'],
 ['ENSG00000103316',
  15,
  '87.67%',
  '83.13%',
  '94.52%',
  '93.65%',
  '88.46%',
  '93.69%',
  '88.39%',
  '0.5997'],
 ['ENSG00000223609',
  16,
  '86.30%',
  '86.75%',
  '88.89%',
  '85.71%',
  '87.80%',
  '93.61%',
  '86.23%',
  '0.4344'],
 ['ENSG00000229124',
  17,
  '86.30%',
  '84.34%',
  '90.91%',
  '88.89%',
  '87.50%',
  '93.46%',
  '86.61%',
  '0.4590'],
 ['ENSG00000213465',
  18,
  '86.99%',
  '87.95%',
  '89.02%',
  '85.71%',
  '88.48%',
  '93.92%',
  '86.83%',
  '0.4832'],
 ['ENSG0

In [ ]:
df_result = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\results_mlp_sfs\mlp_bn_sfs_test.csv", low_memory=False)

In [22]:
df_result2 = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\results_mlp_sfs\mlp_bn_sfs_val.csv", low_memory=False)

In [23]:
df_result2

,Gene,N_Genes,ACC,Sn,Precision,Sp,F1,AUC,BA,Threshold
0,ENSG00000137502,11,87.67%,86.75%,91.14%,88.89%,88.89%,91.78%,87.82%,0.4361
1,ENSG00000177556,12,86.30%,91.57%,85.39%,79.37%,88.37%,93.04%,85.47%,0.2867
2,ENSG00000173083,13,84.25%,77.11%,94.12%,93.65%,84.77%,92.85%,85.38%,0.5308
3,ENSG00000197459,14,84.93%,80.72%,91.78%,90.48%,85.90%,92.96%,85.60%,0.5252
4,ENSG00000103316,15,87.67%,83.13%,94.52%,93.65%,88.46%,93.69%,88.39%,0.5997
...,...,...,...,...,...,...,...,...,...,...
210,ENSG00000115590,207,90.41%,85.54%,97.26%,96.83%,91.03%,97.32%,91.18%,0.7234
211,ENSG00000052749,208,92.47%,91.57%,95.00%,93.65%,93.25%,97.00%,92.61%,0.3494
212,ENSG00000161944,209,90.41%,90.36%,92.59%,90.48%,91.46%,97.04%,90.42%,0.4019
213,ENSG00000129824,210,90.41%,87.95%,94.81%,93.65%,91.25%,96.94%,90.80%,0.6990


In [24]:
for col in ["ACC", "Precision", "Sp", "Sn", "F1", "AUC", "BA"]:
    df_result2[col] = df_result2[col].str.rstrip("%").astype(float)


In [19]:
# Use '&' to ensure BOTH conditions are met
sp_threshold = 0
auc_threshold = 80.0

# This creates a strict filter
mask = (df_result["Sp"] >= sp_threshold) & (df_result["AUC"] >= auc_threshold)
filtered_df = df_result[mask]

if not filtered_df.empty:
    best_row = filtered_df.loc[filtered_df["AUC"].idxmax()]
    print("Best Model Meeting Strict Criteria:")
    print(best_row)
else:
    print(f"No models found with Sp > {sp_threshold} and AUC > {auc_threshold}.")
    # Optional: see what the highest Sp available is
    print(f"Max Sp in data: {df_result['Sp'].max()}")

Best Model Meeting Strict Criteria:
Gene         ENSG00000090013
N_Genes                  195
ACC                    90.16
Sn                     84.76
Precision               97.8
Sp                     97.44
F1                     90.82
AUC                    96.06
BA                      91.1
Threshold             0.3803
Name: 198, dtype: object


In [26]:
df_result2.iloc[198]

Gene         ENSG00000090013
N_Genes                  195
ACC                    91.78
Sn                     91.57
Precision              93.83
Sp                     92.06
F1                     92.68
AUC                    96.84
BA                     91.81
Threshold             0.3803
Name: 198, dtype: object